# **Notebook 3: Baseline Model Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] Internet access (to download the model)
- [ ] GPU runtime enabled (Runtime → Change runtime type → T4 GPU)

**Files this notebook will CREATE:**
- [ ] `outputs.json` — `test_query`, `ground_truth`, `baseline_output` _(Required by NB4, NB5, NB7)_

---

## **Stage 3: Solution V1 (Retrieval-Assisted Generation)**

### **Task 3.1: Establish Baseline Performance**

#### **3.1.1 Execute Baseline Inference [2 marks]**
**The Task:** Load the pre-trained base model in 4-bit quantization and generate a response to an ambiguous shipping-delay query without any context.

**Hints & Tips:**
* Use `do_sample=False` for deterministic output. Do NOT pair `temperature=0.0` with `do_sample=False` — it throws a deprecation warning. Use `temperature=None, top_p=None`.
* `BitsAndBytesConfig(load_in_4bit=True)` shrinks the 1.5B model to ~750MB VRAM.
* `max_new_tokens=120` gives room for a complete answer.

**Model Selection:**
* **Qwen/Qwen2.5-1.5B-Instruct** (recommended) — must match what you used in NB2.
* **TinyLlama-1.1B-Chat** — lighter, weaker structured output.
* **Llama-3-8B-Instruct** — best quality, may OOM on free T4 during fine-tuning.

**Learner Inference:** This establishes your zero-shot baseline. Every later improvement is measured against this exact output.

In [1]:
!pip install -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 45.3 MB/s eta 0:00:00:00:0100:01


In [2]:
# YOUR CODE HERE
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# 4-bit quantisation config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model loaded ✅")
print(f"Device: {next(model.parameters()).device}")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded ✅
Device: cuda:0


In [3]:
# YOUR CODE HERE
def run_inference(query):
    messages = [
        {"role": "system", "content": "You are a customer support intent classifier. Extract the intent and category from the customer query and return a valid JSON object only. No explanation."},
        {"role": "user", "content": query}
    ]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            temperature=None,
            top_p=None
        )
    
    # Decode only the new tokens
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    return response

# Test with ambiguous shipping delay query
query = "I ordered something 2 weeks ago and it still hasn't arrived, this is ridiculous!"
response = run_inference(query)
print(f"Query:    {query}")
print(f"Response: {response}")

Query:    I ordered something 2 weeks ago and it still hasn't arrived, this is ridiculous!
Response: {
  "intent": "complaint",
  "category": "delivery"
}


#### **3.1.2 Evaluate Baseline Quality [2 marks]**
**The Task:** Assess the baseline output for factual inaccuracies against the ground-truth SOP rule.

**Hints & Tips:**
* Compare against the known rule: "Domestic orders deliver within 3-7 business days."
* Did the model invent a timeline? Mention a non-existent tracking system or department?
* Document every hallucination — it justifies Stages 3 and 4.

**Learner Inference:** This hallucination is exactly why you build Stage 3 (a database) and Stage 4 (a router).

In [4]:
# YOUR CODE HERE
import json
import re

# Ground truth SOP rule
SOP_RULE = "Domestic orders deliver within 3-7 business days."

# Test queries to evaluate baseline
test_queries = [
    "I ordered something 2 weeks ago and it still hasn't arrived, this is ridiculous!",
    "Where is my package? It's been 10 days!",
    "My order hasn't arrived yet, can you help?",
    "I need a refund for my order",
    "How do I reset my password?"
]

# Run inference on all queries
results = []
for query in test_queries:
    response = run_inference(query)

    def parse_response(response):
    # Strip ```json ... ``` fences if present
        cleaned = re.sub(r"```json|```", "", response).strip()
        try:
            return json.loads(cleaned), True
        except json.JSONDecodeError:
            return {}, False
    
    # Check format adherence
    try:
        parsed, is_valid_json = parse_response(response)
    except json.JSONDecodeError:
        parsed = {}
        is_valid_json = False
    
    # Check for hallucination — did model invent a timeline or policy?
    hallucination_keywords = ["business days", "hours", "department", "tracking system", "24", "48", "72", "week"]
    has_hallucination = any(kw in response.lower() for kw in hallucination_keywords)
    
    results.append({
        "query": query,
        "response": response,
        "valid_json": is_valid_json,
        "hallucination_detected": has_hallucination
    })
    print(f"Query:       {query[:60]}...")
    print(f"Response:    {response}")
    print(f"Valid JSON:  {is_valid_json}")
    print(f"Hallucination: {has_hallucination}")
    print("-" * 60)

# Summary
valid_count = sum(r['valid_json'] for r in results)
halluc_count = sum(r['hallucination_detected'] for r in results)

print(f"\n--- Baseline Quality Summary ---")
print(f"Format Adherence:      {valid_count}/{len(results)} ({valid_count/len(results)*100:.0f}%)")
print(f"Hallucination Rate:    {halluc_count}/{len(results)} ({halluc_count/len(results)*100:.0f}%)")
print(f"\nGround Truth SOP Rule: {SOP_RULE}")
print("Note: Baseline has no access to SOP — any policy details in output are hallucinated.")

Query:       I ordered something 2 weeks ago and it still hasn't arrived,...
Response:    {
  "intent": "complaint",
  "category": "delivery"
}
Valid JSON:  True
Hallucination: False
------------------------------------------------------------
Query:       Where is my package? It's been 10 days!...
Response:    {
  "intent": "customer_service",
  "category": "package_status"
}
Valid JSON:  True
Hallucination: False
------------------------------------------------------------
Query:       My order hasn't arrived yet, can you help?...
Response:    ```json
{
  "intent": "customer_service",
  "category": "order_status"
}
```
Valid JSON:  True
Hallucination: False
------------------------------------------------------------
Query:       I need a refund for my order...
Response:    ```json
{
  "intent": "refund",
  "category": "customer service"
}
```
Valid JSON:  True
Hallucination: False
------------------------------------------------------------
Query:       How do I reset my password?..

## Baseline Evaluation Findings

**Format Adherence: 100% (5/5) — after code-fence cleaning**
- Raw baseline output: 40% valid (3/5 responses wrapped in ```json fences)
- After applying parse_response() cleaning: 100%
- Note: Production system cannot rely on manual cleaning — fine-tuning must eliminate fences

**Intent Quality: Poor**
- Intents too generic: `complaint`, `customer_service`, `help`, `refund`
- Categories inconsistent and lowercase: `delivery`, `order_status`, `customer service`
- No alignment with Bitext intent schema (e.g., `track_refund`, `cancel_order`)

**Hallucination Rate: 0%**
- Model returned JSON only — no free-text policy claims detected
- Risk increases when model generates full natural language responses

**Conclusion**
Baseline produces parseable JSON only after post-processing.
Intent labels are vague and schema-inconsistent.
This justifies:
- Stage 3 (RAG): Ground responses in corporate SOP documents  
- Stage 4 (Fine-Tuning): Train model to produce schema-aligned intents without post-processing

---
## Save Artifacts for Downstream Notebooks

**IMPORTANT:** Saves the baseline output. Notebooks 4, 5, and 7 depend on this file.

In [6]:
# YOUR CODE HERE
import os
import pandas as pd
test_df = pd.read_csv("/kaggle/input/datasets/piyushshx/artifacts/artifacts/df_test.csv")

In [7]:
import json
import os

os.makedirs("/kaggle/working/artifacts", exist_ok=True)

outputs = []

for _, row in test_df.iterrows():
    response = run_inference(row['instruction'])
    parsed, is_valid = parse_response(response)
    
    outputs.append({
        "test_query":      row['instruction'],
        "ground_truth":    row['intent'],
        "baseline_output": response,
        "parsed_intent":   parsed.get("intent", ""),
        "parsed_category": parsed.get("category", ""),
        "valid_json":      is_valid
    })

# Save as JSON
with open("/kaggle/working/artifacts/outputs.json", "w") as f:
    json.dump(outputs, f, indent=2)

print(f"Saved {len(outputs)} records to outputs.json ✅")

# Verify
with open("/kaggle/working/artifacts/outputs.json") as f:
    check = json.load(f)
print(f"Verified: {len(check)} records loaded back ✅")
print(f"Sample keys: {list(check[0].keys())}")

Saved 391 records to outputs.json ✅
Verified: 391 records loaded back ✅
Sample keys: ['test_query', 'ground_truth', 'baseline_output', 'parsed_intent', 'parsed_category', 'valid_json']


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 4.**

- [ ] Base model loaded in 4-bit without errors
- [ ] Baseline output generated for `test_query`
- [ ] Hallucination assessment documented
- [ ] **`outputs.json` saved** with `test_query`, `ground_truth`, `baseline_output` ← _CRITICAL for NB4, 5, 7_
- [ ] GPU runtime enabled

**If any item is unchecked, fix it before moving on.**